# Optymalizacja bezpośrednia: metoda GPS


Wykorzystaj poniższą metodę `generalized_pattern_search` do optymalizacji funkcji Rosenbrocka. Przetestuj następujące zestawy kierunków:

* `D=[[0, 1], [1, 0], [-1, -1]]`
* `D=[[0, -1], [-1, 0], [1, 1]]`
* `D=[[0, 1], [1, 0], [-1, 0], [0, -1]]`

oraz dwa inne samodzielnie wybrane (proszę pamiętać, że mają dodatnio rozpinać $\mathbb{R}^2$). Wykorzystaj przygotowany wcześniej kod do ewaluacji metod przeszukiwania aby określić na ile dobre są różne zestawy kierunków. Przyjmij $\alpha=1$ oraz $\epsilon=10^{-8}$.

In [2]:
function generalized_pattern_search(f, x, α, D, ε, γ=0.5)
    y, n = f(x), length(x)
    while α > ε
        improved = false
        for (i,d) in enumerate(D)
            x′ = x + α*d
            y′ = f(x′)
            if y′ < y
                x, y, improved = x′, y′, true
                D = pushfirst!(deleteat!(D, i), d)
                break
            end
        end
        if !improved
            α *= γ
        end
    end
    return x
end

generalized_pattern_search (generic function with 2 methods)

In [1]:
function f_rosenbrock(x)
    result = 0.0
    for i in 1:2:length(x)
        result += (1.0 - x[i])^2 + 100.0 * (x[i + 1] - x[i]^2)^2
    end
    return result
end

function g_rosenbrock!(storage, x)
    for i in 1:2:length(x)
        storage[i] = -2.0 * (1.0 - x[i]) - 400.0 * (x[i + 1] - x[i]^2) * x[i]
        storage[i + 1] = 200.0 * (x[i + 1] - x[i]^2)
    end
    return storage
end

g_rosenbrock! (generic function with 1 method)

Porównaj zbieżność z metodą największego spadku.

Spróbuj zoptymalizować metodę `generalized_pattern_search` tak, aby wykonywała jak najmniej alokacji w trakcie działania. Możesz wypróbować:
1. Prealokację `x′` przed pętlą oraz broadcasting w `x′ = x + α*d`.
2. Przepisanie `D = pushfirst!(deleteat!(D, i), d)`.
3. Bibliotekę `StaticArrays.jl`.


In [8]:
using BenchmarkTools

x0 = [0.0, 0.0]
D = [[0, 1], [1, 0], [-1, -1]]
@benchmark generalized_pattern_search($f_rosenbrock, $x0, 1.0, $D, 10^-8)

BenchmarkTools.Trial: 5196 samples with 1 evaluation.
 Range (min … max):  821.044 μs …   2.615 ms  ┊ GC (min … max): 0.00% … 63.61%
 Time  (median):     872.677 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   959.459 μs ± 311.868 μs  ┊ GC (mean ± σ):  8.34% ± 14.29%

  █▇▇▇▆▅▃▁                                                      ▂
  ████████▇▆▅▁▁▁▁▃▁▃▁▁▁▃▁▁▁▁▁▁▁▄▁▁▁▅▅▃▃▃▁▁▁▁▃▆█▇██▆▆▇▆█████▇█▇█ █
  821 μs        Histogram: log(frequency) by time       2.26 ms <

 Memory estimate: 2.86 MiB, allocs estimate: 37550.